# Notebook for exploatory data analysis
Included checks:
- [] max and min transaction timestamps for fetched collections to make sure all records are properly extracted

## Imports & variable assignment

In [34]:
import pandas as pd
from datetime import datetime, timezone
import glob
import os

In [35]:
opensea_preprocessed_data_folder = "data/processed/opensea/"

# Notebook for exploatory data analysis
Included checks:
- [] max and min transaction timestamps for fetched collections to make sure all records are properly extracted

## Imports & variable assignment

In [36]:
import pandas as pd
from datetime import datetime, timezone
import glob
import json
import os

In [37]:
opensea_preprocessed_data_folder = "data/processed/opensea/"

In [39]:
def load_parquet_folder(folder_path: str) -> pd.DataFrame:
    """
    Loads and concatenates all Parquet files from a given folder.

    Assumes folder is located relative to repository root (one level above cwd).

    Returns:
        pd.DataFrame: concatenated dataset from all Parquet files
    """

    # Resolve repository root path
    base_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
    full_path = os.path.join(base_dir, folder_path)

    # Find parquet files
    parquet_files = glob.glob(os.path.join(full_path, "*.parquet"))

    if not parquet_files:
        print(f"⚠️ No Parquet files found in: {full_path}")
        return pd.DataFrame()

    # Load and concatenate
    dfs = [pd.read_parquet(file) for file in parquet_files]
    df = pd.concat(dfs, ignore_index=True)

    print(f"✅ Loaded {len(df)} rows from {len(parquet_files)} files")

    return df

In [40]:
# Load OpenSea NFT transaction dataset
df_raw = load_parquet_folder(opensea_preprocessed_data_folder)

# Quick preview of dataset structure
df_raw.head()

✅ Loaded 4399 rows from 3 files


,event_timestamp,closing_date,transaction,chain,seller,buyer,nft.name,nft.contract,nft.collection,payment.quantity,payment.symbol,payment.decimals,payment.token_address,token_amount,event_day
0,2025-07-28 14:43:47,1753713827,0x9f763994020190b702b13f63c16448456c225e0c9013...,ethereum,0x7d533ac6cbcee97bf00b7ca10650eab713055736,0x084263a11a1c2998ad4d1e5a79ee57d28ff714c2,CryptoPunk #5418,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,5.170000e+19,ETH,18,0x0000000000000000000000000000000000000000,51.70,2025-07-28
1,2025-07-28 11:23:11,1753701791,0x3269f0c515ea67d593aed3c6dc50767c660a45ec6e31...,ethereum,0x799224fdb4e635f5833d66e5ba61ffcd1c1fc558,0x978852db1bf809e30576c951733227d2a8fcc07d,CryptoPunk #5512,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,5.733000e+19,ETH,18,0x0000000000000000000000000000000000000000,57.33,2025-07-28
2,2025-07-28 09:43:23,1753695803,0xed19e54a88fa195c9c75f22d4235a19353358ee37a8d...,ethereum,0x625202e9583039bd61cf599593d269b70521bfa1,0x24fedde1e5e2220256cb1819b7ee7f7b1b20ef5d,CryptoPunk #618,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,5.149000e+19,ETH,18,0x0000000000000000000000000000000000000000,51.49,2025-07-28
3,2025-07-28 08:47:47,1753692467,0x62cd1d149ad8ffbd9853a0df0b524a6a2bb2eb6cf0c4...,ethereum,0x4fc4457788f12aed1909cea1e4eeafaf2b8a4a00,0x8124ce96fbdce2cd3a80f0ba3b124a98a807be38,CryptoPunk #7364,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,5.850000e+19,ETH,18,0x0000000000000000000000000000000000000000,58.50,2025-07-28
4,2025-07-28 06:30:47,1753684247,0x342fc7ac6413262273136c4bba0e7a54243b32012253...,ethereum,0x86f39177283138fd6f5e344dfb78675ba4759ada,0x084263a11a1c2998ad4d1e5a79ee57d28ff714c2,CryptoPunk #5512,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,5.450000e+19,ETH,18,0x0000000000000000000000000000000000000000,54.50,2025-07-28


In [41]:
df_raw.columns

Index(['event_timestamp', 'closing_date', 'transaction', 'chain', 'seller',
       'buyer', 'nft.name', 'nft.contract', 'nft.collection',
       'payment.quantity', 'payment.symbol', 'payment.decimals',
       'payment.token_address', 'token_amount', 'event_day'],
      dtype='str')

In [42]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 4399 entries, 0 to 4398
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   event_timestamp        4399 non-null   datetime64[ms]
 1   closing_date           4399 non-null   int64         
 2   transaction            4399 non-null   str           
 3   chain                  4399 non-null   str           
 4   seller                 4399 non-null   str           
 5   buyer                  4399 non-null   str           
 6   nft.name               4399 non-null   str           
 7   nft.contract           4399 non-null   str           
 8   nft.collection         4399 non-null   str           
 9   payment.quantity       4399 non-null   float64       
 10  payment.symbol         4399 non-null   str           
 11  payment.decimals       4399 non-null   int64         
 12  payment.token_address  4399 non-null   str           
 13  token_amount  

In [43]:
# Create working copy for feature engineering
df = df_raw.copy()

df["event_timestamp"] = pd.to_datetime(df["event_timestamp"], utc=True)

df["event_day"] = df["event_timestamp"].dt.date

df = df.sort_values("event_timestamp").reset_index(drop=True)

# Draft inital Wash Trading rules

## 1. Initial self-trades check

In [48]:
# --------------------------------------------------
# 1. Self-trade detection
# --------------------------------------------------
# Definition: self-trade occurs when seller and buyer are the same wallet.
# This is a direct anomaly signal in NFT wash trading detection.

df["self_trade"] = df["seller"] == df["buyer"]

self_trade_counts = df["self_trade"].value_counts()

print("Self-trade distribution:")
print(self_trade_counts)

Self-trade distribution:
self_trade
False    4391
True        8
Name: count, dtype: int64


In [57]:
# --------------------------------------------------
# Self-trade inspection
# --------------------------------------------------
# Definition: self-trades occur when seller and buyer wallets are identical.

self_trades = df[df["self_trade"] == True]

if self_trades.empty:
    print("No self-trades detected.")
else:
    print(f" 🚨 Detected {len(self_trades)} self-trades")

    display(
        self_trades[
            [
                "event_timestamp",
                "transaction",
                "seller",
                "buyer",
                "nft.name",
                "token_amount",
                "nft.collection",
            ]
        ].head(20)
    )

 🚨 Detected 8 self-trades


,event_timestamp,transaction,seller,buyer,nft.name,token_amount,nft.collection
49,2025-07-02 16:41:11+00:00,0xc85422fd4415de934241e2fd3179eb91dfc5da699ddc...,0xb751379ff5dbc7ae98279486f151172dd37abaa3,0xb751379ff5dbc7ae98279486f151172dd37abaa3,#4049,11.19375,boredapeyachtclub
1307,2025-07-21 16:23:35+00:00,0x4a47f805d07a6bbe91683c65760d1707bf1c44fb7a89...,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#1688,12.83550,boredapeyachtclub
2097,2025-08-02 13:48:11+00:00,0xc8b673ea6dc7fa237f1eed2f2b7396f573a6ac2b550b...,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#640,12.08925,boredapeyachtclub
2488,2025-08-10 01:05:11+00:00,0xa4aea2116accd60f3f14846cba9aabe5eaa7a3953db4...,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,Pudgy Penguin #7077,15.92000,pudgypenguins
2713,2025-08-14 13:36:23+00:00,0x3d970aa61501f3f9d19e2e36f5cd434b4d2542a90881...,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,Pudgy Penguin #3681,13.43250,pudgypenguins
2917,2025-08-18 12:11:35+00:00,0xfc35f2a18ab426d3784de61bb90c57cd870e2db3abcb...,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#5457,12.62655,boredapeyachtclub
3741,2025-09-11 18:25:59+00:00,0x09ecd6a6f589843389f526e46be3d87ee2019cb9c8e7...,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#9494,9.20375,boredapeyachtclub
4366,2025-09-29 02:57:59+00:00,0xe95fb67ee5dee29935c092b3ff34ba139297a77d8af7...,0x3caebdddfb38d801cf4a2ec638085c36abea9c9e,0x3caebdddfb38d801cf4a2ec638085c36abea9c9e,Pudgy Penguin #1915,10.15740,pudgypenguins


## 2. Holding period

In [50]:
# --------------------------------------------------
# 2. Holding period analysis
# --------------------------------------------------
# Definition: holding period measures time difference between consecutive sales
# of the same NFT. Very short holding periods may indicate wash trading.

df["prev_sale_time"] = df.groupby("nft.name")["event_timestamp"].shift(1)

df["holding_seconds"] = (
    df["event_timestamp"] - df["prev_sale_time"]
).dt.total_seconds()

In [58]:
# Thresholds (based on heuristic assumptions from literature / domain intuition)
ONE_DAY = 86_400   # 24 hours
ONE_HOUR = 3_600   # 1 hour

long_holds = df[df["holding_seconds"] > ONE_DAY]

print(f" 🚨 Long holding periods (>24h): {len(long_holds)}")

 🚨 Long holding periods (>24h): 1380


In [59]:
short_holds = df[df["holding_seconds"] <= ONE_HOUR]

print(f" 🚨 Short holding periods (<=1h): {len(short_holds)}")

display(
    short_holds[
        [
            "event_timestamp",
            "prev_sale_time",
            "holding_seconds",
            "seller",
            "buyer",
            "nft.name",
            "token_amount",
        ]
    ].head(20)
)

 🚨 Short holding periods (<=1h): 381


,event_timestamp,prev_sale_time,holding_seconds,seller,buyer,nft.name,token_amount
6,2025-07-01 13:51:47+00:00,2025-07-01 13:45:47+00:00,360.0,0x94e9d73a2aeec807d85756b84759965b41866450,0x621c70de47c35be4622c891555a6016cde2e1a61,Pudgy Penguin #2897,9.598956
18,2025-07-01 21:57:47+00:00,2025-07-01 21:57:47+00:00,0.0,0x5129750f8c36b794e564d3fc5ef01c283d79ec36,0xfc02d4571ba182ac2fdf3e14fcb37f32e79bd28c,#7246,20.000000
48,2025-07-02 16:41:11+00:00,2025-07-02 16:41:11+00:00,0.0,0x7e7022f8879d88bcc5d288b229737adb4b1f39cb,0xb751379ff5dbc7ae98279486f151172dd37abaa3,#4049,10.630000
49,2025-07-02 16:41:11+00:00,2025-07-02 16:41:11+00:00,0.0,0xb751379ff5dbc7ae98279486f151172dd37abaa3,0xb751379ff5dbc7ae98279486f151172dd37abaa3,#4049,11.193750
68,2025-07-03 15:49:47+00:00,2025-07-03 15:39:59+00:00,588.0,0x8bab603fde4824950fa81268a84ef9cbb89cad69,0x843fcb0cf448e714e735eb4b30e81186b9d58ec6,#524,10.740000
78,2025-07-03 19:30:11+00:00,2025-07-03 18:40:47+00:00,2964.0,0xa6a8c4aff47e1a2e907cd1ef5161f14db7e0d9ba,0xf76246b0842c92ad5bd745973ca9eb85b937b126,#9129,10.430000
97,2025-07-04 16:31:59+00:00,2025-07-04 16:30:11+00:00,108.0,0x1fee3385b22d69e93209db2042be58fcac57b59b,0xf2a7be2d334db950b8e490def5412827eac8494c,Pudgy Penguin #5156,9.300000
117,2025-07-07 12:01:11+00:00,2025-07-07 12:00:23+00:00,48.0,0x8bab603fde4824950fa81268a84ef9cbb89cad69,0xf76246b0842c92ad5bd745973ca9eb85b937b126,#3465,11.540000
118,2025-07-07 12:01:59+00:00,2025-07-07 12:01:11+00:00,48.0,0xf76246b0842c92ad5bd745973ca9eb85b937b126,0x66816bd969ff35688e9fd82b3e46fd505260216a,#3465,11.580000
132,2025-07-07 23:20:59+00:00,2025-07-07 22:21:59+00:00,3540.0,0x2463a7b933a50188ff83ab5c703fe6ef3a0449c5,0xf76246b0842c92ad5bd745973ca9eb85b937b126,Pudgy Penguin #936,8.910000


## 3. Initial NFT flip count

In [61]:
# --------------------------------------------------
# 3. NFT flip frequency
# --------------------------------------------------
# Definition: number of transactions per NFT.
# High frequency may indicate artificial trading activity.

df["nft_id"] = df["nft.contract"].astype(str) + "_" + df["nft.name"].astype(str)

flip_counts = df.groupby("nft_id").size()
df["nft_flip_count"] = df["nft_id"].map(flip_counts)

flip_distribution = df["nft_flip_count"].value_counts().sort_index()

print("NFT flip frequency distribution:")
print(flip_distribution)

NFT flip frequency distribution:
nft_flip_count
1     869
2     970
3     693
4     548
5     340
6     348
7     112
8     112
9      99
10     70
11     22
12     24
13     26
14     28
17     17
22     22
99     99
Name: count, dtype: int64


## 4. Wallet-pair behavior (VERY important)
Same pair trades multiple times
Same NFT traded back and forth between same pair

In [62]:
# --------------------------------------------------
# 4. Wallet pair interaction analysis
# --------------------------------------------------
# Definition: measures how often the same wallet pair trades with each other.
# High repetition may indicate coordinated trading behavior.

pair_counts = (
    df.groupby(["seller", "buyer"])
      .size()
      .reset_index(name="pair_trade_count")
)

df = df.merge(pair_counts, on=["seller", "buyer"], how="left")

REPEAT_THRESHOLD = 5

repeated_pairs = df[df["pair_trade_count"] >= REPEAT_THRESHOLD]

print(f" 🚨 High-frequency wallet pairs (>= {REPEAT_THRESHOLD} trades): {len(repeated_pairs)}")

 🚨 High-frequency wallet pairs (>= 5 trades): 76


## Circular trading detection (A → B → A)

In [70]:
# --------------------------------------------------
# Circular trading detection (improved heuristic)
# --------------------------------------------------
# Goal:
# Detect repeated trading cycles between wallets for the same NFT.
# This captures more realistic wash trading behavior than simple A→B→A patterns.

# Step 1: Create directed trading edges per NFT
df = df.sort_values(["nft_id", "event_timestamp"])

# Step 2: Track previous owner for each NFT
df["prev_owner"] = df.groupby("nft_id")["seller"].shift(1)

# Step 3: Detect immediate reversals (A → B → A)
df["immediate_round_trip"] = df["buyer"] == df["prev_owner"]

# Step 4: Detect repeated interactions between same wallet pairs
# (A↔B occurring multiple times for same NFT)
pair_cycle_counts = (
    df.groupby(["nft_id", "seller", "buyer"])
      .size()
      .reset_index(name="pair_repeat_count")
)

df = df.merge(pair_cycle_counts, on=["nft_id", "seller", "buyer"], how="left")

df["repeated_pair_cycle"] = df["pair_repeat_count"] >= 2

# Step 5: Combined circular trading signal
df["circular_trading"] = (
    df["immediate_round_trip"] | df["repeated_pair_cycle"]
)

# Step 6: Extract suspicious circular trades
circular_trades = df[df["circular_trading"]]

print(f"🚨 Detected {len(circular_trades)} circular trading-related transactions")

display(
    circular_trades[
        [
            "event_timestamp",
            "nft.name",
            "seller",
            "buyer",
            "nft_id",
            "immediate_round_trip",
            "pair_repeat_count",
            "token_amount",
        ]
    ].head(20)
)

🚨 Detected 64 circular trading-related transactions


,event_timestamp,nft.name,seller,buyer,nft_id,immediate_round_trip,pair_repeat_count,token_amount
32,2025-09-24 22:05:11+00:00,CryptoPunk #1613,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,0x80bf7db69556d9521c03461978b8fc731dbbd4e4,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,True,1,4.649000e+01
36,2025-07-28 19:36:47+00:00,CryptoPunk #1668,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,0x7650f20becf1d3d37407e281645c638d1ae8939c,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,False,2,5.389000e+01
37,2025-07-28 19:36:59+00:00,CryptoPunk #1668,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,0x7650f20becf1d3d37407e281645c638d1ae8939c,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,False,2,5.389000e+01
101,2025-09-17 21:04:23+00:00,CryptoPunk #267,0xea889aa7ede714a4cff37e81487a3919c28f2e64,0x8b971c66d743ab519d273b2b36de15814f584369,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,True,1,4.749000e+01
111,2025-09-08 18:38:35+00:00,CryptoPunk #2783,0xc50673edb3a7b94e8cad8a7d4e0cd68864e33edf,0xd5ef7d3d225770bcfc4a46f9cef413f440610dee,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,True,1,1.200000e-16
153,2025-09-28 10:30:35+00:00,CryptoPunk #3483,0x3042887f97821ec36be64d2677efd2e943a4cb0f,0x6ec1b656f9ea50c89827c7e820c303a6039550e3,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,True,1,4.495000e+01
279,2025-09-04 01:31:23+00:00,CryptoPunk #5295,0x73f69db86dd39ac8172723d7a150a6aeb316d08a,0x32aa011c1e2ce7aab918e97f5567b871709e654b,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,False,2,4.900000e+01
280,2025-09-04 01:31:47+00:00,CryptoPunk #5295,0x73f69db86dd39ac8172723d7a150a6aeb316d08a,0x32aa011c1e2ce7aab918e97f5567b871709e654b,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,False,2,4.900000e+01
295,2025-09-23 18:24:23+00:00,CryptoPunk #5467,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,0x00000f91109c4d0007e90000d9facad5298a0cac,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,True,1,4.557000e+01
335,2025-09-27 15:45:35+00:00,CryptoPunk #6058,0x0cc3b5015c678685f3552fb1a1bfa53112b71486,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,True,1,4.500000e+01


## 5. Price abnormality (statistical signal)

In [65]:
# --------------------------------------------------
# Price anomaly detection (robust statistical method)
# --------------------------------------------------
# Goal:
# Identify abnormal NFT transaction prices relative to
# their collection-day distribution using robust statistics
# (median + MAD-based z-score).

# Step 1: Compute daily collection statistics
daily_stats = (
    df.groupby(["nft.collection", "event_day"])
      .agg(
          median_price=("token_amount", "median"),
          mad_price=("token_amount", lambda x: (x - x.median()).abs().median()),
          n_trades=("token_amount", "count")
      )
      .reset_index()
)

# Step 2: Keep only statistically reliable groups
# (avoid unstable estimates for very small samples)
daily_stats = daily_stats[daily_stats["n_trades"] >= 5]

# Step 3: Merge statistics back into main dataset
df = df.merge(
    daily_stats,
    on=["nft.collection", "event_day"],
    how="left"
)

# Step 4: Prevent division by zero / numerical instability
df["mad_price"] = df["mad_price"].replace(0, 1e-9)

# Step 5: Compute robust z-score (based on MAD)
df["price_zscore"] = (
    0.6745 *
    (df["token_amount"] - df["median_price"]) /
    df["mad_price"]
)

# Step 6: Flag abnormal prices
PRICE_ZSCORE_THRESHOLD = 3.5
df["price_abnormality"] = df["price_zscore"].abs() > PRICE_ZSCORE_THRESHOLD

# Step 7: Extract anomalies for inspection
abnormal_prices = df[df["price_abnormality"]]

print(f"🚨 Detected {len(abnormal_prices)} price anomalies")

display(
    abnormal_prices[
        [
            "event_timestamp",
            "nft.name",
            "seller",
            "buyer",
            "token_amount",
            "median_price",
            "price_zscore",
        ]
    ].head(20)
)

🚨 Detected 313 price anomalies


,event_timestamp,nft.name,seller,buyer,token_amount,median_price,price_zscore
2,2025-07-01 04:55:35+00:00,CryptoPunk #8688,0x1b32e8c0ca7c479709f0f14c80ab82e44fdc16c8,0x08acffada9db314bd31f9086d0d2dc08e5abf162,41.00000,39.095,4.430767
17,2025-07-01 21:57:47+00:00,#7246,0xae28ac81a65e7a430fa69ed9cd1c199ec7f63d30,0x5129750f8c36b794e564d3fc5ef01c283d79ec36,19.03000,10.900,9.792295
18,2025-07-01 21:57:47+00:00,#7246,0x5129750f8c36b794e564d3fc5ef01c283d79ec36,0xfc02d4571ba182ac2fdf3e14fcb37f32e79bd28c,20.00000,10.900,10.960625
37,2025-07-02 09:41:23+00:00,Pudgy Penguin #2886,0x7fe04fbcbff289cab8b0f2bb305c945f2d36cc0b,0xf4c40bf7070fdcf64ecf020bcb583738a6cc3bcd,12.00000,9.720,11.829692
38,2025-07-02 09:41:47+00:00,Pudgy Penguin #7850,0xcfe122de90515685f806e79b29ff9351ce764098,0xf4c40bf7070fdcf64ecf020bcb583738a6cc3bcd,12.00000,9.720,11.829692
39,2025-07-02 09:48:35+00:00,#8826,0xc848860b09dc8ad9e2f23a4b3a565539639d7fd6,0x264f3508ed91cc3881cd9d29f01b9035d5d84a18,11.68000,10.615,3.504111
52,2025-07-02 16:59:11+00:00,#4206,0x11e5f31fd0e62654912eaea3d403b5a761fb6fd4,0x7d4d1f45a9fec88d1715981088a32b3e6c7cc321,14.50000,10.615,12.782601
59,2025-07-03 02:49:23+00:00,#7360,0x2fd7a107328dd6e5ac618667ff895bbadd7d7935,0x3ed0b065e6fd906ca20e4ebe080ea72c4325339a,13.75000,10.740,9.228386
60,2025-07-03 04:46:59+00:00,Pudgy Penguin #887,0xf76246b0842c92ad5bd745973ca9eb85b937b126,0xc4da1707a7a16cf4ac15d75d6a49a684b23ee1e4,11.36900,9.740,4.777220
63,2025-07-03 10:36:47+00:00,Pudgy Penguin #2921,0x531fb8dc4e65b5bc64def5d4915082ed8fdd8a8c,0xf4c40bf7070fdcf64ecf020bcb583738a6cc3bcd,16.00000,9.740,18.358130


## 6. Final wash trading score

In [71]:
# --------------------------------------------------
# Wash trading risk scoring model
# --------------------------------------------------
# This section computes a heuristic risk score for each transaction
# based on behavioral, temporal, network, and price-based signals.
# The weights are manually defined (expert-driven heuristic approach).

# Step 1: Compute composite wash trading score
df["wash_score"] = (
    df["self_trade"].fillna(False).astype(int) * 35 +
    df["round_trip"].fillna(False).astype(int) * 25 +
    (df["pair_trade_count"].fillna(0) >= 5).astype(int) * 15 +
    (df["holding_seconds"].fillna(0) <= 3600).astype(int) * 10 +
    (df["nft_flip_count"].fillna(0) >= 3).astype(int) * 10 +
    df["price_abnormality"].fillna(False).astype(int) * 5
)

# Step 2: Normalize score range
df["wash_score"] = df["wash_score"].clip(0, 100)

# Step 3: Risk classification function
def classify_score(score):
    if score >= 70:
        return "High Risk"
    elif score >= 30:
        return "Medium Risk"
    else:
        return "Low Risk"

df["wash_risk"] = df["wash_score"].apply(classify_score)

# Step 4: Extract suspicious transactions
suspicious_transactions = df[df["wash_score"] >= 30]

print(f"🚨 Detected {len(suspicious_transactions)} suspicious transactions")

display(
    suspicious_transactions[
        [
            "event_timestamp",
            "nft.name",
            "seller",
            "buyer",
            "token_amount",
            "wash_score",
            "wash_risk",
            "self_trade",
            "round_trip",
            "pair_trade_count",
            "nft_flip_count",
            "price_zscore",
        ]
    ].head(30)
)

🚨 Detected 69 suspicious transactions


,event_timestamp,nft.name,seller,buyer,token_amount,wash_score,wash_risk,self_trade,round_trip,pair_trade_count,nft_flip_count,price_zscore
32,2025-09-24 22:05:11+00:00,CryptoPunk #1613,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,0x80bf7db69556d9521c03461978b8fc731dbbd4e4,4.649000e+01,45,Medium Risk,False,True,2,3,-1.332209
101,2025-09-17 21:04:23+00:00,CryptoPunk #267,0xea889aa7ede714a4cff37e81487a3919c28f2e64,0x8b971c66d743ab519d273b2b36de15814f584369,4.749000e+01,45,Medium Risk,False,True,1,4,NaN
111,2025-09-08 18:38:35+00:00,CryptoPunk #2783,0xc50673edb3a7b94e8cad8a7d4e0cd68864e33edf,0xd5ef7d3d225770bcfc4a46f9cef413f440610dee,1.200000e-16,35,Medium Risk,False,True,1,2,NaN
153,2025-09-28 10:30:35+00:00,CryptoPunk #3483,0x3042887f97821ec36be64d2677efd2e943a4cb0f,0x6ec1b656f9ea50c89827c7e820c303a6039550e3,4.495000e+01,35,Medium Risk,False,True,1,2,-0.674500
295,2025-09-23 18:24:23+00:00,CryptoPunk #5467,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,0x00000f91109c4d0007e90000d9facad5298a0cac,4.557000e+01,35,Medium Risk,False,True,1,2,NaN
335,2025-09-27 15:45:35+00:00,CryptoPunk #6058,0x0cc3b5015c678685f3552fb1a1bfa53112b71486,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,4.500000e+01,35,Medium Risk,False,True,1,3,0.720185
343,2025-09-29 16:07:11+00:00,CryptoPunk #6140,0x24ac005adcbb3d9551474b43f855191e12e8a4ec,0x6ec1b656f9ea50c89827c7e820c303a6039550e3,4.608500e+01,35,Medium Risk,False,True,1,2,-1.089844
406,2025-08-14 18:40:47+00:00,CryptoPunk #6741,0xf4c40bf7070fdcf64ecf020bcb583738a6cc3bcd,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,5.000000e+01,35,Medium Risk,False,True,2,5,0.004003
407,2025-08-14 19:16:47+00:00,CryptoPunk #6741,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,0xf4c40bf7070fdcf64ecf020bcb583738a6cc3bcd,5.000000e+01,45,Medium Risk,False,True,3,5,0.004003
413,2025-09-14 16:56:23+00:00,CryptoPunk #6821,0xfb7bda74654ae653a17806794fbcf4d9fafb2153,0x3cee9254b2ee12a949a899dc35417a1862b74391,4.800000e+01,35,Medium Risk,False,True,1,2,0.589063


In [72]:
suspicious_rate = len(suspicious_transactions) / len(df)
print(suspicious_rate)

0.015685383041600363


In [69]:
df["wash_risk"].value_counts()

wash_risk
Low Risk       4330
Medium Risk      63
High Risk         6
Name: count, dtype: int64

## Wallet interaction graph

In [ ]:
# Build edge list
edges = df[["seller", "buyer", "nft_id", "event_timestamp"]].copy()
display(edges)

,seller,buyer,nft_id,event_timestamp
0,0x87d411bb593218f43b1c76278769a0f5d1b740f4,0x12c9bbec46f5f808044f01b865e4f953d1a3d727,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d_#5956,2025-06-30 23:15:11
1,0x8c2b143b0276bafd2613bb41d7058583cf6706c7,0x2777df13d272c9433858f5e1947b51e6a4720d60,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d_#4733,2025-07-01 03:02:35
2,0x1b32e8c0ca7c479709f0f14c80ab82e44fdc16c8,0x08acffada9db314bd31f9086d0d2dc08e5abf162,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,2025-07-01 04:55:35
3,0x7f41c4d5fad289d1b788a8a94d38e180445b7724,0xf2a7be2d334db950b8e490def5412827eac8494c,0xbd3531da5cf5857e7cfaa92426877b022e612cf8_Pud...,2025-07-01 07:13:23
4,0x9c0a836384a2f28dcd658ed77fd655caabd719a5,0xe5c3a3bad46475dd53100cdbecb0a7541aba0391,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,2025-07-01 12:48:59
...,...,...,...,...
4394,0xaa130e5a43cadbe2eaa4e58ede61e39874e3ead7,0xad51b811695ebef57540e241ba98d5ec071a849d,0xbd3531da5cf5857e7cfaa92426877b022e612cf8_Pud...,2025-09-29 19:19:47
4395,0xf4fb6a00cdb1f39af1a6900eedf8598fa9533c32,0xe594096eb1beb1a19b03b2e6b343bd29a9d9f618,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d_#7412,2025-09-29 20:29:47
4396,0xeb79f3b9836e14678abdb862aabfc171341b3199,0xcfbf3aa72bcc8af0ba064d9e7b83888495a280de,0xbd3531da5cf5857e7cfaa92426877b022e612cf8_Pud...,2025-09-29 20:38:47
4397,0x8e8d6246c45d0e7f68172e85573546d90fc2e062,0xc5f8b6f97cef80ad7eb11924a57dae3cb938c555,0xbd3531da5cf5857e7cfaa92426877b022e612cf8_Pud...,2025-09-29 20:51:47


In [ ]:
# Compute wallet interaction counts
seller_degree = df.groupby("seller")["buyer"].nunique()
buyer_degree = df.groupby("buyer")["seller"].nunique()

In [ ]:
# Total interaction intensity
wallet_activity = df.groupby("seller").size() + df.groupby("buyer").size()
print(wallet_activity)

seller
0x000000000000fea5f4b241f9e77b4d43b76798a9    NaN
0x0000000188e3604489698ea73de28524f2bea6c6    NaN
0x00000f91109c4d0007e90000d9facad5298a0cac    4.0
0x0025c38da57bf4eafcb7d31770716232dee5ced2    NaN
0x00422818436951ab9abf04defdccecd9c6ad429d    NaN
                                             ... 
0xff7c2c3d6bdefcad650100b96a74e0e7b0fe3041    NaN
0xffb285456daa57d757b9c9020a6302afe5a95ed6    NaN
0xffb90de6e35b6a93cbcf823a1d286b07f0adb64b    NaN
0xffbd08371fae1e00e28be21d7a274790ba2dbaf6    NaN
0xffec68dced77787469f1abc902f1d04b9cc09866    NaN
Length: 2266, dtype: float64


In [ ]:
df["low_counterparty_diversity"] = df["seller"].map(seller_degree) < 3

df["wallet_network_risk"] = (
    df["low_counterparty_diversity"].astype(int) * 10
)

In [ ]:
df["wash_score"] += df["wallet_network_risk"]

In [ ]:
display(df)

,event_timestamp,closing_date,transaction,chain,seller,buyer,nft.name,nft.contract,nft.collection,payment.quantity,...,prev_seller,round_trip,median_price,mad_price,price_zscore,price_abnormality,wash_score,wash_risk,low_counterparty_diversity,wallet_network_risk
0,2025-06-30 23:15:11,1751325311,0xbb6584c252f53522eb83f1b591e96adac4af1b2225f6...,ethereum,0x87d411bb593218f43b1c76278769a0f5d1b740f4,0x12c9bbec46f5f808044f01b865e4f953d1a3d727,#5956,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,1.111000e+19,...,NaN,False,11.110000,1.000000e-09,0.000000,False,10,Low Risk,True,10
1,2025-07-01 03:02:35,1751338955,0xb38e864a9b8cc51adad2fa03c5d59bf3a276e64209e4...,ethereum,0x8c2b143b0276bafd2613bb41d7058583cf6706c7,0x2777df13d272c9433858f5e1947b51e6a4720d60,#4733,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,1.067877e+19,...,NaN,False,10.900000,5.600000e-01,-0.266463,False,20,Low Risk,True,10
2,2025-07-01 04:55:35,1751345735,0x69c62b56f7661e5129e966cd3700c1fc2926ea474f4d...,ethereum,0x1b32e8c0ca7c479709f0f14c80ab82e44fdc16c8,0x08acffada9db314bd31f9086d0d2dc08e5abf162,CryptoPunk #8688,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,4.100000e+19,...,NaN,False,39.095000,2.900000e-01,4.430767,True,15,Low Risk,True,10
3,2025-07-01 07:13:23,1751354003,0x7f3a5edc6bf624d71c1be9910d15f70e716f92904d83...,ethereum,0x7f41c4d5fad289d1b788a8a94d38e180445b7724,0xf2a7be2d334db950b8e490def5412827eac8494c,Pudgy Penguin #5859,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,pudgypenguins,9.597000e+18,...,NaN,False,9.573500,1.244778e-01,0.127338,False,0,Low Risk,False,0
4,2025-07-01 12:48:59,1751374139,0xc4fc7cc0defd760a48d3e814db219cc090824eae194a...,ethereum,0x9c0a836384a2f28dcd658ed77fd655caabd719a5,0xe5c3a3bad46475dd53100cdbecb0a7541aba0391,CryptoPunk #8151,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,3.920000e+19,...,NaN,False,39.095000,2.900000e-01,0.244216,False,10,Low Risk,True,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4394,2025-09-29 19:19:47,1759173587,0xb76c923ee73678231db46aaabafc10906132bd8d5cff...,ethereum,0xaa130e5a43cadbe2eaa4e58ede61e39874e3ead7,0xad51b811695ebef57540e241ba98d5ec071a849d,Pudgy Penguin #7938,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,pudgypenguins,1.233000e+19,...,0xdec8a93d4a8924b2f5362ba57d726282a6cf262e,False,10.235000,5.050000e-02,27.981733,True,15,Low Risk,True,10
4395,2025-09-29 20:29:47,1759177787,0xd56bb98ca2bf168a688b782b726f738defd8d3efcebd...,ethereum,0xf4fb6a00cdb1f39af1a6900eedf8598fa9533c32,0xe594096eb1beb1a19b03b2e6b343bd29a9d9f618,#7412,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,8.950000e+18,...,0x0d2eb2be84e68497e2eb8c84b72597450f1b23cc,False,9.129989,2.049893e-01,-0.592240,False,10,Low Risk,False,0
4396,2025-09-29 20:38:47,1759178327,0x8345b1e9b236f3eed1a9aa038b64cfaf622d4f2e33d0...,ethereum,0xeb79f3b9836e14678abdb862aabfc171341b3199,0xcfbf3aa72bcc8af0ba064d9e7b83888495a280de,Pudgy Penguin #1492,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,pudgypenguins,1.007000e+19,...,NaN,False,10.235000,5.050000e-02,-2.203812,False,0,Low Risk,False,0
4397,2025-09-29 20:51:47,1759179107,0xda2fb141ffb49807debe5ceed8ceed06109c29f8fb15...,ethereum,0x8e8d6246c45d0e7f68172e85573546d90fc2e062,0xc5f8b6f97cef80ad7eb11924a57dae3cb938c555,Pudgy Penguin #36,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,pudgypenguins,1.022000e+19,...,0x53c801b4a0ef5e747e86f7e7c1564c57c80d8e9d,False,10.235000,5.050000e-02,-0.200347,False,10,Low Risk,False,0


In [ ]:
wash_trades = df[
    (df["wash_score"] >= 70)
]
display(wash_trades)

,event_timestamp,closing_date,transaction,chain,seller,buyer,nft.name,nft.contract,nft.collection,payment.quantity,...,prev_seller,round_trip,median_price,mad_price,price_zscore,price_abnormality,wash_score,wash_risk,low_counterparty_diversity,wallet_network_risk
1307,2025-07-21 16:23:35,1753115015,0x4a47f805d07a6bbe91683c65760d1707bf1c44fb7a89...,ethereum,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#1688,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,1.283550e+19,...,0x8bab603fde4824950fa81268a84ef9cbb89cad69,False,12.699500,0.42050,0.218150,False,70,High Risk,False,0
2097,2025-08-02 13:48:11,1754142491,0xc8b673ea6dc7fa237f1eed2f2b7396f573a6ac2b550b...,ethereum,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#640,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,1.208925e+19,...,0x54d28b26487e6529cf447434bccf439bea352e5d,True,12.089250,0.47925,0.000000,False,95,High Risk,False,0
2488,2025-08-10 01:05:11,1754787911,0xa4aea2116accd60f3f14846cba9aabe5eaa7a3953db4...,ethereum,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,Pudgy Penguin #7077,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,pudgypenguins,1.592000e+19,...,0xd43dfa9024096b0224adf6859e22fbc35ea2b21f,False,14.969000,0.23900,2.683889,False,70,High Risk,False,0
2713,2025-08-14 13:36:23,1755178583,0x3d970aa61501f3f9d19e2e36f5cd434b4d2542a90881...,ethereum,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,Pudgy Penguin #3681,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,pudgypenguins,1.343250e+19,...,0x54d28b26487e6529cf447434bccf439bea352e5d,True,13.150000,0.12000,1.587885,False,95,High Risk,False,0
2917,2025-08-18 12:11:35,1755519095,0xfc35f2a18ab426d3784de61bb90c57cd870e2db3abcb...,ethereum,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#5457,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,1.262655e+19,...,0x54d28b26487e6529cf447434bccf439bea352e5d,True,11.797800,0.70780,0.789760,False,95,High Risk,False,0
3741,2025-09-11 18:25:59,1757615159,0x09ecd6a6f589843389f526e46be3d87ee2019cb9c8e7...,ethereum,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#9494,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,9.203750e+18,...,0x6469561e2e92a611ce967de5ca310a119f106c2e,False,9.226875,0.03500,-0.445652,False,70,High Risk,False,0


In [ ]:
wash_trades_2 = df[
    (df["self_trade"]) |
    (df["round_trip"]) |
    (df["pair_trade_count"] >= 5)
]

display(wash_trades_2)

,event_timestamp,closing_date,transaction,chain,seller,buyer,nft.name,nft.contract,nft.collection,payment.quantity,...,prev_seller,round_trip,median_price,mad_price,price_zscore,price_abnormality,wash_score,wash_risk,low_counterparty_diversity,wallet_network_risk
20,2025-07-02 00:59:11,1751417951,0x77145357d610ee41ca938b01d21b721dbc70991a8596...,ethereum,0xd8f24f5f0382e197c1e87ad82b357209383470cf,0xede4e6d73a5277910a6d5459d255e4d2095a2373,Pudgy Penguin #3134,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,pudgypenguins,9.420100e+18,...,0x8f8f7ce7472c5017b71af3e445ee2242130f9f8c,False,9.720000,0.130000,-1.556020,False,25,Low Risk,False,0
44,2025-07-02 15:29:47,1751470187,0xa874196107fc1b21a52cc98ae4e8d53b7e979bceb896...,ethereum,0xf76246b0842c92ad5bd745973ca9eb85b937b126,0x195f46025a6926968a1b3275822096eb12d97e70,#6209,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,1.044000e+19,...,0x82115fa4148355d6e937b1250076976721fcb113,False,10.615000,0.205000,-0.575793,False,25,Low Risk,False,0
48,2025-07-02 16:41:11,1751474471,0xc85422fd4415de934241e2fd3179eb91dfc5da699ddc...,ethereum,0x7e7022f8879d88bcc5d288b229737adb4b1f39cb,0xb751379ff5dbc7ae98279486f151172dd37abaa3,#4049,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,1.063000e+19,...,0xb751379ff5dbc7ae98279486f151172dd37abaa3,True,10.615000,0.205000,0.049354,False,45,Medium Risk,False,0
49,2025-07-02 16:41:11,1751474471,0xc85422fd4415de934241e2fd3179eb91dfc5da699ddc...,ethereum,0xb751379ff5dbc7ae98279486f151172dd37abaa3,0xb751379ff5dbc7ae98279486f151172dd37abaa3,#4049,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,1.119375e+19,...,0x7e7022f8879d88bcc5d288b229737adb4b1f39cb,False,10.615000,0.205000,1.904229,False,65,Medium Risk,True,10
148,2025-07-09 00:38:11,1752021491,0x6a61d0d7d92c13448c797b83999624efb6e4237208a1...,ethereum,0x621c70de47c35be4622c891555a6016cde2e1a61,0x94e9d73a2aeec807d85756b84759965b41866450,Pudgy Penguin #207,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,pudgypenguins,9.089990e+18,...,0x94e9d73a2aeec807d85756b84759965b41866450,True,8.720000,0.140000,1.782559,False,35,Low Risk,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4266,2025-09-27 15:45:35,1758987935,0x4dd390d206d763c9f7627387c50225c3a019cbc488bd...,ethereum,0x0cc3b5015c678685f3552fb1a1bfa53112b71486,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,CryptoPunk #6058,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,4.500000e+19,...,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,True,44.802549,0.184925,0.720185,False,35,Low Risk,False,0
4275,2025-09-27 16:27:59,1758990479,0xe45a908f876fe4d7188dffb258b3c13ccf9d49daa664...,ethereum,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,0x80bf7db69556d9521c03461978b8fc731dbbd4e4,CryptoPunk #9763,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,4.480000e+19,...,0x80bf7db69556d9521c03461978b8fc731dbbd4e4,True,44.802549,0.184925,-0.009299,False,45,Medium Risk,False,0
4321,2025-09-28 10:30:35,1759055435,0xd666bb745b9413ebf3e324fd4db7ec9556b498907242...,ethereum,0x3042887f97821ec36be64d2677efd2e943a4cb0f,0x6ec1b656f9ea50c89827c7e820c303a6039550e3,CryptoPunk #3483,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,4.495000e+19,...,0x6ec1b656f9ea50c89827c7e820c303a6039550e3,True,45.000000,0.050000,-0.674500,False,45,Low Risk,True,10
4366,2025-09-29 02:57:59,1759114679,0xe95fb67ee5dee29935c092b3ff34ba139297a77d8af7...,ethereum,0x3caebdddfb38d801cf4a2ec638085c36abea9c9e,0x3caebdddfb38d801cf4a2ec638085c36abea9c9e,Pudgy Penguin #1915,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,pudgypenguins,1.015740e+19,...,NaN,False,10.235000,0.050500,-1.036459,False,45,Low Risk,True,10


In [6]:
opensea_data_df = load_parquet_folder(folder_path=opensea_preprocessed_data_folder)
opensea_data_df.head()

/Users/juliaprzygoda/Desktop/thesis-code/nft-wash-trading-detection/data/processed/opensea/
✅ Found 3 Parquet files in data/processed/opensea/
📊 DataFrame created with 4399 rows and 15 columns from data/processed/opensea/


,event_timestamp,closing_date,transaction,chain,seller,buyer,nft.name,nft.contract,nft.collection,payment.quantity,payment.symbol,payment.decimals,payment.token_address,token_amount,event_day
0,2025-07-28 14:43:47,1753713827,0x9f763994020190b702b13f63c16448456c225e0c9013...,ethereum,0x7d533ac6cbcee97bf00b7ca10650eab713055736,0x084263a11a1c2998ad4d1e5a79ee57d28ff714c2,CryptoPunk #5418,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,5.170000e+19,ETH,18,0x0000000000000000000000000000000000000000,51.70,2025-07-28
1,2025-07-28 11:23:11,1753701791,0x3269f0c515ea67d593aed3c6dc50767c660a45ec6e31...,ethereum,0x799224fdb4e635f5833d66e5ba61ffcd1c1fc558,0x978852db1bf809e30576c951733227d2a8fcc07d,CryptoPunk #5512,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,5.733000e+19,ETH,18,0x0000000000000000000000000000000000000000,57.33,2025-07-28
2,2025-07-28 09:43:23,1753695803,0xed19e54a88fa195c9c75f22d4235a19353358ee37a8d...,ethereum,0x625202e9583039bd61cf599593d269b70521bfa1,0x24fedde1e5e2220256cb1819b7ee7f7b1b20ef5d,CryptoPunk #618,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,5.149000e+19,ETH,18,0x0000000000000000000000000000000000000000,51.49,2025-07-28
3,2025-07-28 08:47:47,1753692467,0x62cd1d149ad8ffbd9853a0df0b524a6a2bb2eb6cf0c4...,ethereum,0x4fc4457788f12aed1909cea1e4eeafaf2b8a4a00,0x8124ce96fbdce2cd3a80f0ba3b124a98a807be38,CryptoPunk #7364,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,5.850000e+19,ETH,18,0x0000000000000000000000000000000000000000,58.50,2025-07-28
4,2025-07-28 06:30:47,1753684247,0x342fc7ac6413262273136c4bba0e7a54243b32012253...,ethereum,0x86f39177283138fd6f5e344dfb78675ba4759ada,0x084263a11a1c2998ad4d1e5a79ee57d28ff714c2,CryptoPunk #5512,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,5.450000e+19,ETH,18,0x0000000000000000000000000000000000000000,54.50,2025-07-28


In [7]:
opensea_data_df.columns

Index(['event_timestamp', 'closing_date', 'transaction', 'chain', 'seller',
       'buyer', 'nft.name', 'nft.contract', 'nft.collection',
       'payment.quantity', 'payment.symbol', 'payment.decimals',
       'payment.token_address', 'token_amount', 'event_day'],
      dtype='str')

In [8]:
df = opensea_data_df.copy()

# timestamp sanity
df["event_timestamp"] = pd.to_datetime(df["event_timestamp"], utc=True)

# required for daily stats
df["event_day"] = df["event_timestamp"].dt.date

# Draft inital Wash Trading rules

## 1. Initial self-trades check

In [10]:
opensea_data_df["self_trade"] = (
    opensea_data_df["seller"] == opensea_data_df["buyer"]
)

self_trade_counts = opensea_data_df["self_trade"].value_counts()

print(self_trade_counts)

self_trade
False    4391
True        8
Name: count, dtype: int64


In [11]:
self_trades = opensea_data_df[opensea_data_df["self_trade"]]

if self_trades.empty:
    print("✅ No self-trades found")
else:
    print(f"🚨 Found {len(self_trades)} self-trades")
    display(
        self_trades[
            [
                "event_timestamp",
                "transaction",
                "seller",
                "buyer",
                "nft.name",
                "token_amount",
                "nft.collection",
            ]
        ]
    )

🚨 Found 8 self-trades


,event_timestamp,transaction,seller,buyer,nft.name,token_amount,nft.collection
935,2025-08-10 01:05:11,0xa4aea2116accd60f3f14846cba9aabe5eaa7a3953db4...,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,Pudgy Penguin #7077,15.92000,pudgypenguins
1157,2025-08-14 13:36:23,0x3d970aa61501f3f9d19e2e36f5cd434b4d2542a90881...,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,Pudgy Penguin #3681,13.43250,pudgypenguins
2169,2025-09-29 02:57:59,0xe95fb67ee5dee29935c092b3ff34ba139297a77d8af7...,0x3caebdddfb38d801cf4a2ec638085c36abea9c9e,0x3caebdddfb38d801cf4a2ec638085c36abea9c9e,Pudgy Penguin #1915,10.15740,pudgypenguins
3121,2025-09-11 18:25:59,0x09ecd6a6f589843389f526e46be3d87ee2019cb9c8e7...,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#9494,9.20375,boredapeyachtclub
3250,2025-07-21 16:23:35,0x4a47f805d07a6bbe91683c65760d1707bf1c44fb7a89...,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#1688,12.83550,boredapeyachtclub
3306,2025-08-02 13:48:11,0xc8b673ea6dc7fa237f1eed2f2b7396f573a6ac2b550b...,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#640,12.08925,boredapeyachtclub
3682,2025-07-02 16:41:11,0xc85422fd4415de934241e2fd3179eb91dfc5da699ddc...,0xb751379ff5dbc7ae98279486f151172dd37abaa3,0xb751379ff5dbc7ae98279486f151172dd37abaa3,#4049,11.19375,boredapeyachtclub
4035,2025-08-18 12:11:35,0xfc35f2a18ab426d3784de61bb90c57cd870e2db3abcb...,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#5457,12.62655,boredapeyachtclub


## 2. Holding period

In [12]:
df = opensea_data_df.sort_values("event_timestamp")

df["prev_sale_time"] = (
    df.groupby("nft.name")["event_timestamp"].shift(1)
)

df["holding_seconds"] = (
    df["event_timestamp"] - df["prev_sale_time"]
).dt.total_seconds()


THRESHOLD = 86_400  # 1 day

long_holds = df[df["holding_seconds"] > THRESHOLD]

print(f"Found {len(long_holds)} records above threshold")
display(
    long_holds[
        [
            "event_timestamp",
            "prev_sale_time",
            "holding_seconds",
            "seller",
            "buyer",
            "nft.name",
            "token_amount",
        ]
    ].head(20)
)

Found 1380 records above threshold


,event_timestamp,prev_sale_time,holding_seconds,seller,buyer,nft.name,token_amount
3686,2025-07-02 15:29:47,2025-07-01 14:30:59,89928.0,0xf76246b0842c92ad5bd745973ca9eb85b937b126,0x195f46025a6926968a1b3275822096eb12d97e70,#6209,10.440000
2366,2025-07-03 13:53:47,2025-07-02 12:18:11,92136.0,0x96e4a596087b811d61216684a4d87c688c6cbffa,0x6b8bc451c1d54ea4b67d2b3fdea9acd865f06eed,Pudgy Penguin #4691,9.840000
2363,2025-07-03 18:25:35,2025-07-02 04:52:23,135192.0,0xd5c4e4ed8486404df0b22a8f72a440e8d59fedba,0x8b291046edd260252fa00642f299144580ffe596,Pudgy Penguin #6612,9.970000
2362,2025-07-03 18:28:35,2025-07-01 16:47:35,178860.0,0xf2a7be2d334db950b8e490def5412827eac8494c,0x83735f0f1c732b28d044f22279a4f97218582dec,Pudgy Penguin #2897,9.740000
2360,2025-07-03 18:29:23,2025-07-02 05:58:59,131424.0,0x16f41a775a4ffc5da7b465597abfcb8ec0469d28,0x83735f0f1c732b28d044f22279a4f97218582dec,Pudgy Penguin #4792,9.699000
2359,2025-07-03 20:03:35,2025-07-02 00:59:11,155064.0,0xede4e6d73a5277910a6d5459d255e4d2095a2373,0xa6a8c4aff47e1a2e907cd1ef5161f14db7e0d9ba,Pudgy Penguin #3134,9.699990
3661,2025-07-04 16:24:23,2025-07-02 15:29:47,176076.0,0x195f46025a6926968a1b3275822096eb12d97e70,0xb2bb7767c46e385515c07ffaa68d1db79b3c845e,#6209,10.550000
3658,2025-07-05 06:07:11,2025-07-03 18:46:47,127224.0,0x6dcf1d3259c8a188e265ff72da5bd90f656df566,0xabe620ac5612535ced48f439f6d5232cf6e7ed88,#8997,11.290000
3654,2025-07-06 00:10:47,2025-07-01 03:02:35,421692.0,0x2777df13d272c9433858f5e1947b51e6a4720d60,0x53418545f4a97bb705eae9655213866d416207d5,#4733,11.777000
3652,2025-07-06 06:00:11,2025-07-03 19:30:11,210600.0,0x1ea27bce786a81022dfc156059771e8d3279a9a6,0x492da0e704a8e61fc08073e3b6cbafef4155593c,#9129,10.784103


In [13]:
ONE_HOUR = 60 * 60  # 3600 seconds

short_holds = df[df["holding_seconds"] <= ONE_HOUR]

print(f"🚨 Found {len(short_holds)} trades held ≤ 1 hour")

display(
    short_holds[
        [
            "event_timestamp",
            "prev_sale_time",
            "holding_seconds",
            "seller",
            "buyer",
            "nft.name",
            "token_amount",
        ]
    ].head(20)
)

🚨 Found 381 trades held ≤ 1 hour


,event_timestamp,prev_sale_time,holding_seconds,seller,buyer,nft.name,token_amount
2395,2025-07-01 13:51:47,2025-07-01 13:45:47,360.0,0x94e9d73a2aeec807d85756b84759965b41866450,0x621c70de47c35be4622c891555a6016cde2e1a61,Pudgy Penguin #2897,9.598956
3693,2025-07-01 21:57:47,2025-07-01 21:57:47,0.0,0x5129750f8c36b794e564d3fc5ef01c283d79ec36,0xfc02d4571ba182ac2fdf3e14fcb37f32e79bd28c,#7246,20.000000
3681,2025-07-02 16:41:11,2025-07-02 16:41:11,0.0,0x7e7022f8879d88bcc5d288b229737adb4b1f39cb,0xb751379ff5dbc7ae98279486f151172dd37abaa3,#4049,10.630000
3682,2025-07-02 16:41:11,2025-07-02 16:41:11,0.0,0xb751379ff5dbc7ae98279486f151172dd37abaa3,0xb751379ff5dbc7ae98279486f151172dd37abaa3,#4049,11.193750
3671,2025-07-03 15:49:47,2025-07-03 15:39:59,588.0,0x8bab603fde4824950fa81268a84ef9cbb89cad69,0x843fcb0cf448e714e735eb4b30e81186b9d58ec6,#524,10.740000
3667,2025-07-03 19:30:11,2025-07-03 18:40:47,2964.0,0xa6a8c4aff47e1a2e907cd1ef5161f14db7e0d9ba,0xf76246b0842c92ad5bd745973ca9eb85b937b126,#9129,10.430000
2350,2025-07-04 16:31:59,2025-07-04 16:30:11,108.0,0x1fee3385b22d69e93209db2042be58fcac57b59b,0xf2a7be2d334db950b8e490def5412827eac8494c,Pudgy Penguin #5156,9.300000
3896,2025-07-07 12:01:11,2025-07-07 12:00:23,48.0,0x8bab603fde4824950fa81268a84ef9cbb89cad69,0xf76246b0842c92ad5bd745973ca9eb85b937b126,#3465,11.540000
3895,2025-07-07 12:01:59,2025-07-07 12:01:11,48.0,0xf76246b0842c92ad5bd745973ca9eb85b937b126,0x66816bd969ff35688e9fd82b3e46fd505260216a,#3465,11.580000
2235,2025-07-07 23:20:59,2025-07-07 22:21:59,3540.0,0x2463a7b933a50188ff83ab5c703fe6ef3a0449c5,0xf76246b0842c92ad5bd745973ca9eb85b937b126,Pudgy Penguin #936,8.910000


## 3. Initial NFT flip count

In [14]:
# df["nft_id"] = (
#     opensea_data_df["nft.contract"] + "_" + opensea_data_df["nft.name"]
# )

# flip_counts = df.groupby("nft_id").size()
# df["nft_flip_count"] = df["nft_id"].map(flip_counts)

df["nft_id"] = df["nft.contract"].astype(str) + "_" + df["nft.name"].astype(str)

flip_counts = df.groupby("nft_id").size()
df["nft_flip_count"] = df["nft_id"].map(flip_counts)

# The ones fliped more often can be sus
# We need as well some time conditions
flip_distribution = df["nft_flip_count"].value_counts().sort_index()
print(flip_distribution)

nft_flip_count
1     869
2     970
3     693
4     548
5     340
6     348
7     112
8     112
9      99
10     70
11     22
12     24
13     26
14     28
17     17
22     22
99     99
Name: count, dtype: int64


## 4. Wallet-pair behavior (VERY important)
Same pair trades multiple times
Same NFT traded back and forth between same pair

In [15]:
pair_counts = (
    df.groupby(["seller", "buyer"])
      .size()
      .reset_index(name="pair_trade_count")
)

df = df.merge(pair_counts, on=["seller", "buyer"], how="left")

REPEAT_THRESHOLD = 5

repeated_pairs = df[df["pair_trade_count"] >= REPEAT_THRESHOLD]

print(f"🚨 Found {len(repeated_pairs)} trades where the same seller/buyer pair traded ≥ {REPEAT_THRESHOLD} times")
display(
    repeated_pairs[
        [
            "event_timestamp",
            "seller",
            "buyer",
            "nft.name",
            "nft_id",
            "nft_flip_count",
            "pair_trade_count",
            "holding_seconds",
            "self_trade",
            "token_amount",
        ]
    ].head(20)
)

🚨 Found 76 trades where the same seller/buyer pair traded ≥ 5 times


,event_timestamp,seller,buyer,nft.name,nft_id,nft_flip_count,pair_trade_count,holding_seconds,self_trade,token_amount
20,2025-07-02 00:59:11,0xd8f24f5f0382e197c1e87ad82b357209383470cf,0xede4e6d73a5277910a6d5459d255e4d2095a2373,Pudgy Penguin #3134,0xbd3531da5cf5857e7cfaa92426877b022e612cf8_Pud...,6,9,26508.0,False,9.420100
44,2025-07-02 15:29:47,0xf76246b0842c92ad5bd745973ca9eb85b937b126,0x195f46025a6926968a1b3275822096eb12d97e70,#6209,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d_#6209,9,6,89928.0,False,10.440000
443,2025-07-12 11:40:23,0xd43dfa9024096b0224adf6859e22fbc35ea2b21f,0x3bb4fa84b120ac0dbb4a6bb0442fe2c47e324a93,Pudgy Penguin #1115,0xbd3531da5cf5857e7cfaa92426877b022e612cf8_Pud...,1,5,NaN,False,11.900000
540,2025-07-13 13:12:59,0xff53e1da7b67ae676d7742f858aab5bd4bc937f6,0x5b468edb7688e9ae6c1fa5a6d2debbef06e92907,#1049,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d_#1049,6,14,960.0,False,11.579999
721,2025-07-15 00:23:35,0xf76246b0842c92ad5bd745973ca9eb85b937b126,0x195f46025a6926968a1b3275822096eb12d97e70,,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d_,99,6,5772.0,False,11.490000
845,2025-07-16 18:20:47,0xd8f24f5f0382e197c1e87ad82b357209383470cf,0xede4e6d73a5277910a6d5459d255e4d2095a2373,Pudgy Penguin #6196,0xbd3531da5cf5857e7cfaa92426877b022e612cf8_Pud...,2,9,NaN,False,14.520100
958,2025-07-19 12:54:47,0xd43dfa9024096b0224adf6859e22fbc35ea2b21f,0x3bb4fa84b120ac0dbb4a6bb0442fe2c47e324a93,Pudgy Penguin #6732,0xbd3531da5cf5857e7cfaa92426877b022e612cf8_Pud...,4,5,509844.0,False,14.300000
1016,2025-07-20 18:52:35,0xa25803ab86a327786bb59395fc0164d826b98298,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,CryptoPunk #5918,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,1,7,NaN,False,47.999000
1030,2025-07-20 21:17:23,0x0232d1083e970f0c78f56202b9a666b526fa379f,0x8be240e8689547f1068a835d14f1d943958095dc,CryptoPunk #4398,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,1,13,NaN,False,47.000000
1034,2025-07-20 21:17:23,0x0232d1083e970f0c78f56202b9a666b526fa379f,0x8be240e8689547f1068a835d14f1d943958095dc,CryptoPunk #4656,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,1,13,NaN,False,47.000000


## Circular trading detection (A → B → A)

In [16]:
opensea_data_df["nft.id"] = opensea_data_df["nft.contract"] + "_" + opensea_data_df["nft.name"]

In [18]:
# df = opensea_data_df
# Previous seller for the same NFT
df["prev_seller"] = df.groupby("nft_id")["seller"].shift(1)

# Did the NFT go back to the previous seller? (round-trip)
df["round_trip"] = df["buyer"] == df["prev_seller"]

In [19]:
round_trips = df[df["round_trip"]]

print(f"🚨 Found {len(round_trips)} round-trip trades")
display(
    round_trips[
        [
            "event_timestamp",
            "nft.name",
            "nft.contract",
            "seller",
            "buyer",
            "prev_seller",
            "token_amount",
            # "holding_seconds",
            "self_trade",
            # "pair_trade_count",
        ]
    ].head(20)
)

🚨 Found 47 round-trip trades


,event_timestamp,nft.name,nft.contract,seller,buyer,prev_seller,token_amount,self_trade
48,2025-07-02 16:41:11,#4049,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,0x7e7022f8879d88bcc5d288b229737adb4b1f39cb,0xb751379ff5dbc7ae98279486f151172dd37abaa3,0xb751379ff5dbc7ae98279486f151172dd37abaa3,10.63000,False
148,2025-07-09 00:38:11,Pudgy Penguin #207,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,0x621c70de47c35be4622c891555a6016cde2e1a61,0x94e9d73a2aeec807d85756b84759965b41866450,0x94e9d73a2aeec807d85756b84759965b41866450,9.08999,False
175,2025-07-10 02:18:47,Pudgy Penguin #207,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,0x94e9d73a2aeec807d85756b84759965b41866450,0x621c70de47c35be4622c891555a6016cde2e1a61,0x621c70de47c35be4622c891555a6016cde2e1a61,8.90000,False
344,2025-07-11 18:36:11,#655,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,0x6dcf1d3259c8a188e265ff72da5bd90f656df566,0x8279b7167c17d11fe60238248eef4aa9e750b2b6,0x8279b7167c17d11fe60238248eef4aa9e750b2b6,11.19000,False
727,2025-07-15 05:13:23,Pudgy Penguin #4051,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,0x3bb4fa84b120ac0dbb4a6bb0442fe2c47e324a93,0xb52fc39a77a5681015e77fbf4e6c0e960acc40fd,0xb52fc39a77a5681015e77fbf4e6c0e960acc40fd,14.05000,False
749,2025-07-15 14:26:59,Pudgy Penguin #2082,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,0x1a72af678fc3fb503c4a76bd00e472cb7675bd60,0x146448fb756ed766fe4f87f7a0a1d636d6de8d92,0x146448fb756ed766fe4f87f7a0a1d636d6de8d92,14.00000,False
955,2025-07-19 08:47:47,Pudgy Penguin #79,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,0x1ea27bce786a81022dfc156059771e8d3279a9a6,0x4ce0f96c459df322df68f393569549d5a54a1929,0x4ce0f96c459df322df68f393569549d5a54a1929,13.90000,False
1128,2025-07-20 22:55:23,#8407,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,0x489d9dfbce0986ed4f21a3398c12bbbbff197a6d,0x922cc241ee295ec37c048d5e3e2247bc54883998,0x922cc241ee295ec37c048d5e3e2247bc54883998,11.58500,False
1352,2025-07-22 07:04:35,Pudgy Penguin #4289,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,0xfbc73b89ebc31a7dd27af6fa2a45685821ed5110,0x1ea27bce786a81022dfc156059771e8d3279a9a6,0x1ea27bce786a81022dfc156059771e8d3279a9a6,16.00000,False
1387,2025-07-22 14:42:11,Pudgy Penguin #4289,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,0x1ea27bce786a81022dfc156059771e8d3279a9a6,0xfbc73b89ebc31a7dd27af6fa2a45685821ed5110,0xfbc73b89ebc31a7dd27af6fa2a45685821ed5110,16.00000,False


## 5. Price abnormality (statistical signal)

In [20]:
# Daily collection statistics
daily_stats = (
    df.groupby(["nft.collection", "event_day"])
      .agg(
          median_price=("token_amount", "median"),
          mad_price=("token_amount",
                     lambda x: (x - x.median()).abs().median())
      )
      .reset_index()
)

# Merge back
df = df.merge(
    daily_stats,
    on=["nft.collection", "event_day"],
    how="left"
)

# Avoid division by zero
df["mad_price"] = df["mad_price"].replace(0, 1e-9)

# Robust z-score
df["price_zscore"] = (
    0.6745 *
    (df["token_amount"] - df["median_price"]) /
    df["mad_price"]
)

# Suspicious price jumps
PRICE_ZSCORE_THRESHOLD = 3.5

df["price_abnormality"] = (
    df["price_zscore"].abs() > PRICE_ZSCORE_THRESHOLD
)

abnormal_prices = df[df["price_abnormality"]]

print(f"🚨 Found {len(abnormal_prices)} suspicious price anomalies")

display(
    abnormal_prices[
        [
            "event_timestamp",
            "nft.name",
            "seller",
            "buyer",
            "token_amount",
            "median_price",
            "price_zscore",
        ]
    ].head(20)
)

🚨 Found 330 suspicious price anomalies


,event_timestamp,nft.name,seller,buyer,token_amount,median_price,price_zscore
2,2025-07-01 04:55:35,CryptoPunk #8688,0x1b32e8c0ca7c479709f0f14c80ab82e44fdc16c8,0x08acffada9db314bd31f9086d0d2dc08e5abf162,41.00000,39.095,4.430767
17,2025-07-01 21:57:47,#7246,0xae28ac81a65e7a430fa69ed9cd1c199ec7f63d30,0x5129750f8c36b794e564d3fc5ef01c283d79ec36,19.03000,10.900,9.792295
18,2025-07-01 21:57:47,#7246,0x5129750f8c36b794e564d3fc5ef01c283d79ec36,0xfc02d4571ba182ac2fdf3e14fcb37f32e79bd28c,20.00000,10.900,10.960625
37,2025-07-02 09:41:23,Pudgy Penguin #2886,0x7fe04fbcbff289cab8b0f2bb305c945f2d36cc0b,0xf4c40bf7070fdcf64ecf020bcb583738a6cc3bcd,12.00000,9.720,11.829692
38,2025-07-02 09:41:47,Pudgy Penguin #7850,0xcfe122de90515685f806e79b29ff9351ce764098,0xf4c40bf7070fdcf64ecf020bcb583738a6cc3bcd,12.00000,9.720,11.829692
39,2025-07-02 09:48:35,#8826,0xc848860b09dc8ad9e2f23a4b3a565539639d7fd6,0x264f3508ed91cc3881cd9d29f01b9035d5d84a18,11.68000,10.615,3.504111
52,2025-07-02 16:59:11,#4206,0x11e5f31fd0e62654912eaea3d403b5a761fb6fd4,0x7d4d1f45a9fec88d1715981088a32b3e6c7cc321,14.50000,10.615,12.782601
59,2025-07-03 02:49:23,#7360,0x2fd7a107328dd6e5ac618667ff895bbadd7d7935,0x3ed0b065e6fd906ca20e4ebe080ea72c4325339a,13.75000,10.740,9.228386
60,2025-07-03 04:46:59,Pudgy Penguin #887,0xf76246b0842c92ad5bd745973ca9eb85b937b126,0xc4da1707a7a16cf4ac15d75d6a49a684b23ee1e4,11.36900,9.740,4.777220
63,2025-07-03 10:36:47,Pudgy Penguin #2921,0x531fb8dc4e65b5bc64def5d4915082ed8fdd8a8c,0xf4c40bf7070fdcf64ecf020bcb583738a6cc3bcd,16.00000,9.740,18.358130


## 6. Final wash trading score

In [21]:
df["wash_score"] = (
    df["self_trade"].astype(int) * 35 +
    df["round_trip"].astype(int) * 25 +
    (df["pair_trade_count"] >= 5).astype(int) * 15 +
    (df["holding_seconds"] <= 3600).astype(int) * 10 +
    (df["nft_flip_count"] >= 3).astype(int) * 10 +
    df["price_abnormality"].astype(int) * 5
)

# Normalize to max 100
df["wash_score"] = df["wash_score"].clip(0, 100)

# Risk labels
def classify_score(score):
    if score >= 70:
        return "High Risk"
    elif score >= 40:
        return "Medium Risk"
    else:
        return "Low Risk"

df["wash_risk"] = df["wash_score"].apply(classify_score)

# Suspicious transactions
suspicious_transactions = df[df["wash_score"] >= 40]

print(f"🚨 Found {len(suspicious_transactions)} suspicious transactions")

display(
    suspicious_transactions[
        [
            "event_timestamp",
            "nft.name",
            "seller",
            "buyer",
            "token_amount",
            "wash_score",
            "wash_risk",
            "self_trade",
            "round_trip",
            "pair_trade_count",
            "nft_flip_count",
            "price_zscore",
        ]
    ].head(30)
)

🚨 Found 17 suspicious transactions


,event_timestamp,nft.name,seller,buyer,token_amount,wash_score,wash_risk,self_trade,round_trip,pair_trade_count,nft_flip_count,price_zscore
48,2025-07-02 16:41:11,#4049,0x7e7022f8879d88bcc5d288b229737adb4b1f39cb,0xb751379ff5dbc7ae98279486f151172dd37abaa3,10.630000,45,Medium Risk,False,True,1,7,0.049354
49,2025-07-02 16:41:11,#4049,0xb751379ff5dbc7ae98279486f151172dd37abaa3,0xb751379ff5dbc7ae98279486f151172dd37abaa3,11.193750,55,Medium Risk,True,False,1,7,1.904229
955,2025-07-19 08:47:47,Pudgy Penguin #79,0x1ea27bce786a81022dfc156059771e8d3279a9a6,0x4ce0f96c459df322df68f393569549d5a54a1929,13.900000,45,Medium Risk,False,True,3,10,-0.064939
1128,2025-07-20 22:55:23,#8407,0x489d9dfbce0986ed4f21a3398c12bbbbff197a6d,0x922cc241ee295ec37c048d5e3e2247bc54883998,11.585000,45,Medium Risk,False,True,1,7,0.530754
1307,2025-07-21 16:23:35,#1688,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,12.835500,70,High Risk,True,False,6,7,0.218150
1590,2025-07-25 06:08:11,#300,0x3e6e8b955e6f5484fc95200737e4d092b35c577c,0xf76246b0842c92ad5bd745973ca9eb85b937b126,12.100000,45,Medium Risk,False,True,1,6,0.367909
2097,2025-08-02 13:48:11,#640,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,12.089250,95,High Risk,True,True,6,6,0.000000
2488,2025-08-10 01:05:11,Pudgy Penguin #7077,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,15.920000,70,High Risk,True,False,6,6,2.683889
2713,2025-08-14 13:36:23,Pudgy Penguin #3681,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,13.432500,95,High Risk,True,True,6,4,1.587885
2735,2025-08-14 19:16:47,CryptoPunk #6741,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,0xf4c40bf7070fdcf64ecf020bcb583738a6cc3bcd,50.000000,45,Medium Risk,False,True,3,5,0.004003


## Wallet interaction graph

In [23]:
# Build edge list
edges = df[["seller", "buyer", "nft_id", "event_timestamp"]].copy()
display(edges)

,seller,buyer,nft_id,event_timestamp
0,0x87d411bb593218f43b1c76278769a0f5d1b740f4,0x12c9bbec46f5f808044f01b865e4f953d1a3d727,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d_#5956,2025-06-30 23:15:11
1,0x8c2b143b0276bafd2613bb41d7058583cf6706c7,0x2777df13d272c9433858f5e1947b51e6a4720d60,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d_#4733,2025-07-01 03:02:35
2,0x1b32e8c0ca7c479709f0f14c80ab82e44fdc16c8,0x08acffada9db314bd31f9086d0d2dc08e5abf162,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,2025-07-01 04:55:35
3,0x7f41c4d5fad289d1b788a8a94d38e180445b7724,0xf2a7be2d334db950b8e490def5412827eac8494c,0xbd3531da5cf5857e7cfaa92426877b022e612cf8_Pud...,2025-07-01 07:13:23
4,0x9c0a836384a2f28dcd658ed77fd655caabd719a5,0xe5c3a3bad46475dd53100cdbecb0a7541aba0391,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,2025-07-01 12:48:59
...,...,...,...,...
4394,0xaa130e5a43cadbe2eaa4e58ede61e39874e3ead7,0xad51b811695ebef57540e241ba98d5ec071a849d,0xbd3531da5cf5857e7cfaa92426877b022e612cf8_Pud...,2025-09-29 19:19:47
4395,0xf4fb6a00cdb1f39af1a6900eedf8598fa9533c32,0xe594096eb1beb1a19b03b2e6b343bd29a9d9f618,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d_#7412,2025-09-29 20:29:47
4396,0xeb79f3b9836e14678abdb862aabfc171341b3199,0xcfbf3aa72bcc8af0ba064d9e7b83888495a280de,0xbd3531da5cf5857e7cfaa92426877b022e612cf8_Pud...,2025-09-29 20:38:47
4397,0x8e8d6246c45d0e7f68172e85573546d90fc2e062,0xc5f8b6f97cef80ad7eb11924a57dae3cb938c555,0xbd3531da5cf5857e7cfaa92426877b022e612cf8_Pud...,2025-09-29 20:51:47


In [24]:
# Compute wallet interaction counts
seller_degree = df.groupby("seller")["buyer"].nunique()
buyer_degree = df.groupby("buyer")["seller"].nunique()

In [26]:
# Total interaction intensity
wallet_activity = df.groupby("seller").size() + df.groupby("buyer").size()
print(wallet_activity)

seller
0x000000000000fea5f4b241f9e77b4d43b76798a9    NaN
0x0000000188e3604489698ea73de28524f2bea6c6    NaN
0x00000f91109c4d0007e90000d9facad5298a0cac    4.0
0x0025c38da57bf4eafcb7d31770716232dee5ced2    NaN
0x00422818436951ab9abf04defdccecd9c6ad429d    NaN
                                             ... 
0xff7c2c3d6bdefcad650100b96a74e0e7b0fe3041    NaN
0xffb285456daa57d757b9c9020a6302afe5a95ed6    NaN
0xffb90de6e35b6a93cbcf823a1d286b07f0adb64b    NaN
0xffbd08371fae1e00e28be21d7a274790ba2dbaf6    NaN
0xffec68dced77787469f1abc902f1d04b9cc09866    NaN
Length: 2266, dtype: float64


In [27]:
df["low_counterparty_diversity"] = df["seller"].map(seller_degree) < 3

df["wallet_network_risk"] = (
    df["low_counterparty_diversity"].astype(int) * 10
)

In [28]:
df["wash_score"] += df["wallet_network_risk"]

In [29]:
display(df)

,event_timestamp,closing_date,transaction,chain,seller,buyer,nft.name,nft.contract,nft.collection,payment.quantity,...,prev_seller,round_trip,median_price,mad_price,price_zscore,price_abnormality,wash_score,wash_risk,low_counterparty_diversity,wallet_network_risk
0,2025-06-30 23:15:11,1751325311,0xbb6584c252f53522eb83f1b591e96adac4af1b2225f6...,ethereum,0x87d411bb593218f43b1c76278769a0f5d1b740f4,0x12c9bbec46f5f808044f01b865e4f953d1a3d727,#5956,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,1.111000e+19,...,NaN,False,11.110000,1.000000e-09,0.000000,False,10,Low Risk,True,10
1,2025-07-01 03:02:35,1751338955,0xb38e864a9b8cc51adad2fa03c5d59bf3a276e64209e4...,ethereum,0x8c2b143b0276bafd2613bb41d7058583cf6706c7,0x2777df13d272c9433858f5e1947b51e6a4720d60,#4733,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,1.067877e+19,...,NaN,False,10.900000,5.600000e-01,-0.266463,False,20,Low Risk,True,10
2,2025-07-01 04:55:35,1751345735,0x69c62b56f7661e5129e966cd3700c1fc2926ea474f4d...,ethereum,0x1b32e8c0ca7c479709f0f14c80ab82e44fdc16c8,0x08acffada9db314bd31f9086d0d2dc08e5abf162,CryptoPunk #8688,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,4.100000e+19,...,NaN,False,39.095000,2.900000e-01,4.430767,True,15,Low Risk,True,10
3,2025-07-01 07:13:23,1751354003,0x7f3a5edc6bf624d71c1be9910d15f70e716f92904d83...,ethereum,0x7f41c4d5fad289d1b788a8a94d38e180445b7724,0xf2a7be2d334db950b8e490def5412827eac8494c,Pudgy Penguin #5859,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,pudgypenguins,9.597000e+18,...,NaN,False,9.573500,1.244778e-01,0.127338,False,0,Low Risk,False,0
4,2025-07-01 12:48:59,1751374139,0xc4fc7cc0defd760a48d3e814db219cc090824eae194a...,ethereum,0x9c0a836384a2f28dcd658ed77fd655caabd719a5,0xe5c3a3bad46475dd53100cdbecb0a7541aba0391,CryptoPunk #8151,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,3.920000e+19,...,NaN,False,39.095000,2.900000e-01,0.244216,False,10,Low Risk,True,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4394,2025-09-29 19:19:47,1759173587,0xb76c923ee73678231db46aaabafc10906132bd8d5cff...,ethereum,0xaa130e5a43cadbe2eaa4e58ede61e39874e3ead7,0xad51b811695ebef57540e241ba98d5ec071a849d,Pudgy Penguin #7938,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,pudgypenguins,1.233000e+19,...,0xdec8a93d4a8924b2f5362ba57d726282a6cf262e,False,10.235000,5.050000e-02,27.981733,True,15,Low Risk,True,10
4395,2025-09-29 20:29:47,1759177787,0xd56bb98ca2bf168a688b782b726f738defd8d3efcebd...,ethereum,0xf4fb6a00cdb1f39af1a6900eedf8598fa9533c32,0xe594096eb1beb1a19b03b2e6b343bd29a9d9f618,#7412,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,8.950000e+18,...,0x0d2eb2be84e68497e2eb8c84b72597450f1b23cc,False,9.129989,2.049893e-01,-0.592240,False,10,Low Risk,False,0
4396,2025-09-29 20:38:47,1759178327,0x8345b1e9b236f3eed1a9aa038b64cfaf622d4f2e33d0...,ethereum,0xeb79f3b9836e14678abdb862aabfc171341b3199,0xcfbf3aa72bcc8af0ba064d9e7b83888495a280de,Pudgy Penguin #1492,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,pudgypenguins,1.007000e+19,...,NaN,False,10.235000,5.050000e-02,-2.203812,False,0,Low Risk,False,0
4397,2025-09-29 20:51:47,1759179107,0xda2fb141ffb49807debe5ceed8ceed06109c29f8fb15...,ethereum,0x8e8d6246c45d0e7f68172e85573546d90fc2e062,0xc5f8b6f97cef80ad7eb11924a57dae3cb938c555,Pudgy Penguin #36,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,pudgypenguins,1.022000e+19,...,0x53c801b4a0ef5e747e86f7e7c1564c57c80d8e9d,False,10.235000,5.050000e-02,-0.200347,False,10,Low Risk,False,0


In [30]:
wash_trades = df[
    (df["wash_score"] >= 70)
]
display(wash_trades)

,event_timestamp,closing_date,transaction,chain,seller,buyer,nft.name,nft.contract,nft.collection,payment.quantity,...,prev_seller,round_trip,median_price,mad_price,price_zscore,price_abnormality,wash_score,wash_risk,low_counterparty_diversity,wallet_network_risk
1307,2025-07-21 16:23:35,1753115015,0x4a47f805d07a6bbe91683c65760d1707bf1c44fb7a89...,ethereum,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#1688,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,1.283550e+19,...,0x8bab603fde4824950fa81268a84ef9cbb89cad69,False,12.699500,0.42050,0.218150,False,70,High Risk,False,0
2097,2025-08-02 13:48:11,1754142491,0xc8b673ea6dc7fa237f1eed2f2b7396f573a6ac2b550b...,ethereum,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#640,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,1.208925e+19,...,0x54d28b26487e6529cf447434bccf439bea352e5d,True,12.089250,0.47925,0.000000,False,95,High Risk,False,0
2488,2025-08-10 01:05:11,1754787911,0xa4aea2116accd60f3f14846cba9aabe5eaa7a3953db4...,ethereum,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,Pudgy Penguin #7077,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,pudgypenguins,1.592000e+19,...,0xd43dfa9024096b0224adf6859e22fbc35ea2b21f,False,14.969000,0.23900,2.683889,False,70,High Risk,False,0
2713,2025-08-14 13:36:23,1755178583,0x3d970aa61501f3f9d19e2e36f5cd434b4d2542a90881...,ethereum,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,Pudgy Penguin #3681,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,pudgypenguins,1.343250e+19,...,0x54d28b26487e6529cf447434bccf439bea352e5d,True,13.150000,0.12000,1.587885,False,95,High Risk,False,0
2917,2025-08-18 12:11:35,1755519095,0xfc35f2a18ab426d3784de61bb90c57cd870e2db3abcb...,ethereum,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#5457,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,1.262655e+19,...,0x54d28b26487e6529cf447434bccf439bea352e5d,True,11.797800,0.70780,0.789760,False,95,High Risk,False,0
3741,2025-09-11 18:25:59,1757615159,0x09ecd6a6f589843389f526e46be3d87ee2019cb9c8e7...,ethereum,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#9494,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,9.203750e+18,...,0x6469561e2e92a611ce967de5ca310a119f106c2e,False,9.226875,0.03500,-0.445652,False,70,High Risk,False,0


In [31]:
wash_trades_2 = df[
    (df["self_trade"]) |
    (df["round_trip"]) |
    (df["pair_trade_count"] >= 5)
]

display(wash_trades_2)

,event_timestamp,closing_date,transaction,chain,seller,buyer,nft.name,nft.contract,nft.collection,payment.quantity,...,prev_seller,round_trip,median_price,mad_price,price_zscore,price_abnormality,wash_score,wash_risk,low_counterparty_diversity,wallet_network_risk
20,2025-07-02 00:59:11,1751417951,0x77145357d610ee41ca938b01d21b721dbc70991a8596...,ethereum,0xd8f24f5f0382e197c1e87ad82b357209383470cf,0xede4e6d73a5277910a6d5459d255e4d2095a2373,Pudgy Penguin #3134,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,pudgypenguins,9.420100e+18,...,0x8f8f7ce7472c5017b71af3e445ee2242130f9f8c,False,9.720000,0.130000,-1.556020,False,25,Low Risk,False,0
44,2025-07-02 15:29:47,1751470187,0xa874196107fc1b21a52cc98ae4e8d53b7e979bceb896...,ethereum,0xf76246b0842c92ad5bd745973ca9eb85b937b126,0x195f46025a6926968a1b3275822096eb12d97e70,#6209,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,1.044000e+19,...,0x82115fa4148355d6e937b1250076976721fcb113,False,10.615000,0.205000,-0.575793,False,25,Low Risk,False,0
48,2025-07-02 16:41:11,1751474471,0xc85422fd4415de934241e2fd3179eb91dfc5da699ddc...,ethereum,0x7e7022f8879d88bcc5d288b229737adb4b1f39cb,0xb751379ff5dbc7ae98279486f151172dd37abaa3,#4049,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,1.063000e+19,...,0xb751379ff5dbc7ae98279486f151172dd37abaa3,True,10.615000,0.205000,0.049354,False,45,Medium Risk,False,0
49,2025-07-02 16:41:11,1751474471,0xc85422fd4415de934241e2fd3179eb91dfc5da699ddc...,ethereum,0xb751379ff5dbc7ae98279486f151172dd37abaa3,0xb751379ff5dbc7ae98279486f151172dd37abaa3,#4049,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,1.119375e+19,...,0x7e7022f8879d88bcc5d288b229737adb4b1f39cb,False,10.615000,0.205000,1.904229,False,65,Medium Risk,True,10
148,2025-07-09 00:38:11,1752021491,0x6a61d0d7d92c13448c797b83999624efb6e4237208a1...,ethereum,0x621c70de47c35be4622c891555a6016cde2e1a61,0x94e9d73a2aeec807d85756b84759965b41866450,Pudgy Penguin #207,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,pudgypenguins,9.089990e+18,...,0x94e9d73a2aeec807d85756b84759965b41866450,True,8.720000,0.140000,1.782559,False,35,Low Risk,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4266,2025-09-27 15:45:35,1758987935,0x4dd390d206d763c9f7627387c50225c3a019cbc488bd...,ethereum,0x0cc3b5015c678685f3552fb1a1bfa53112b71486,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,CryptoPunk #6058,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,4.500000e+19,...,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,True,44.802549,0.184925,0.720185,False,35,Low Risk,False,0
4275,2025-09-27 16:27:59,1758990479,0xe45a908f876fe4d7188dffb258b3c13ccf9d49daa664...,ethereum,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,0x80bf7db69556d9521c03461978b8fc731dbbd4e4,CryptoPunk #9763,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,4.480000e+19,...,0x80bf7db69556d9521c03461978b8fc731dbbd4e4,True,44.802549,0.184925,-0.009299,False,45,Medium Risk,False,0
4321,2025-09-28 10:30:35,1759055435,0xd666bb745b9413ebf3e324fd4db7ec9556b498907242...,ethereum,0x3042887f97821ec36be64d2677efd2e943a4cb0f,0x6ec1b656f9ea50c89827c7e820c303a6039550e3,CryptoPunk #3483,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,4.495000e+19,...,0x6ec1b656f9ea50c89827c7e820c303a6039550e3,True,45.000000,0.050000,-0.674500,False,45,Low Risk,True,10
4366,2025-09-29 02:57:59,1759114679,0xe95fb67ee5dee29935c092b3ff34ba139297a77d8af7...,ethereum,0x3caebdddfb38d801cf4a2ec638085c36abea9c9e,0x3caebdddfb38d801cf4a2ec638085c36abea9c9e,Pudgy Penguin #1915,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,pudgypenguins,1.015740e+19,...,NaN,False,10.235000,0.050500,-1.036459,False,45,Low Risk,True,10


In [73]:
# --------------------------------------------------
# Wallet interaction network features
# --------------------------------------------------
# Goal:
# Capture behavioral patterns of wallets based on their interaction diversity.
# Low diversity may indicate coordinated or controlled trading behavior.

# Step 1: Build basic interaction edge list (for interpretability)
edges = df[["seller", "buyer", "nft_id", "event_timestamp"]].copy()

# Step 2: Compute wallet connectivity (network degree approximation)
seller_degree = df.groupby("seller")["buyer"].nunique()
buyer_degree = df.groupby("buyer")["seller"].nunique()

# Step 3: Compute interaction intensity (activity level proxy)
wallet_activity = (
    df.groupby("seller").size() +
    df.groupby("buyer").size()
)

# Step 4: Feature - low counterparty diversity
# (wallet interacts with very few unique counterparties)
df["low_counterparty_diversity"] = df["seller"].map(seller_degree) < 3

# Step 5: Network-based risk signal
df["wallet_network_risk"] = df["low_counterparty_diversity"].astype(int) * 10

# Step 6: Incorporate into wash score (explicitly documented)
df["wash_score"] = df["wash_score"] + df["wallet_network_risk"]

# Step 7: Recompute risk labels (important after score update)
df["wash_score"] = df["wash_score"].clip(0, 100)

def classify_score(score):
    if score >= 70:
        return "High Risk"
    elif score >= 30:
        return "Medium Risk"
    else:
        return "Low Risk"

df["wash_risk"] = df["wash_score"].apply(classify_score)

# Step 8: Final outputs
wash_trades = df[df["wash_score"] >= 70]
wash_trades_weak = df[
    (df["self_trade"]) |
    (df["round_trip"]) |
    (df["pair_trade_count"] >= 5)
]

print(f"High-risk transactions: {len(wash_trades)}")
print(f"Rule-based suspicious transactions: {len(wash_trades_weak)}")

display(wash_trades.head(20))

High-risk transactions: 6
Rule-based suspicious transactions: 122


,event_timestamp,closing_date,transaction,chain,seller,buyer,nft.name,nft.contract,nft.collection,payment.quantity,...,price_abnormality,wash_score,wash_risk,prev_owner,immediate_round_trip,pair_repeat_count,repeated_pair_cycle,circular_trading,low_counterparty_diversity,wallet_network_risk
824,2025-07-21 16:23:35+00:00,1753115015,0x4a47f805d07a6bbe91683c65760d1707bf1c44fb7a89...,ethereum,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#1688,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,1.283550e+19,...,False,70,High Risk,0x8bab603fde4824950fa81268a84ef9cbb89cad69,False,1,False,False,False,0
1353,2025-08-18 12:11:35+00:00,1755519095,0xfc35f2a18ab426d3784de61bb90c57cd870e2db3abcb...,ethereum,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#5457,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,1.262655e+19,...,False,95,High Risk,0x54d28b26487e6529cf447434bccf439bea352e5d,True,1,False,True,False,0
1507,2025-08-02 13:48:11+00:00,1754142491,0xc8b673ea6dc7fa237f1eed2f2b7396f573a6ac2b550b...,ethereum,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#640,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,1.208925e+19,...,False,95,High Risk,0x54d28b26487e6529cf447434bccf439bea352e5d,True,1,False,True,False,0
1972,2025-09-11 18:25:59+00:00,1757615159,0x09ecd6a6f589843389f526e46be3d87ee2019cb9c8e7...,ethereum,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#9494,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d,boredapeyachtclub,9.203750e+18,...,False,70,High Risk,0x6469561e2e92a611ce967de5ca310a119f106c2e,False,1,False,False,False,0
2913,2025-08-14 13:36:23+00:00,1755178583,0x3d970aa61501f3f9d19e2e36f5cd434b4d2542a90881...,ethereum,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,Pudgy Penguin #3681,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,pudgypenguins,1.343250e+19,...,False,95,High Risk,0x54d28b26487e6529cf447434bccf439bea352e5d,True,1,False,True,False,0
3841,2025-08-10 01:05:11+00:00,1754787911,0xa4aea2116accd60f3f14846cba9aabe5eaa7a3953db4...,ethereum,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,Pudgy Penguin #7077,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,pudgypenguins,1.592000e+19,...,False,70,High Risk,0xd43dfa9024096b0224adf6859e22fbc35ea2b21f,False,1,False,False,False,0
